# E4 / RQ3 — FIXED checkpoint persistence (v2)

**Bản sửa từ notebook E4 nhóm đã chạy, đối chiếu với `03_E1_Full_Training.ipynb`.** E1 dùng symlink `encoder/checkpoint/<model>` trỏ trực tiếp tới Google Drive. Bản này áp dụng cùng nguyên tắc cho **từng model × layer × seed**, tránh ghi đè.

- **KHÔNG XÓA** bất kỳ run cũ nào ở `experiments/E4/runs`.
- Các lượt chạy phục hồi ghi vào `experiments/E4/recovery_runs/...`; không thay đổi log/status gốc.
- Checkpoint được **ghi thẳng vào Drive trong quá trình training**, không phụ thuộc copy sau khi train.
- Nếu checkpoint chưa tồn tại hoặc không đọc được: **FAIL**, giữ workspace để debug; không đánh dấu thành công.
- Mặc định `DRY_RUN=True`, `SMOKE_TEST_ONLY=True`. Test một seed trước, sau đó mới chạy tiếp.
- Chạy trên **Google Colab GPU**. Chỉ chỉnh `MODEL`, `LAYER`, `DRY_RUN`, `SMOKE_TEST_ONLY` ở Cell 2.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os, sys, json, csv, shutil, subprocess, hashlib, time
from datetime import datetime, timezone
import pandas as pd, yaml
ROOT=Path('/content/drive/MyDrive/Data_SNA_Group14')
DATA=ROOT/'data/raw/amazon'
E4=ROOT/'experiments/E4'
SOURCE=Path('/content/RLMRec_E4_source')
REPO_URL='https://github.com/HKUDS/RLMRec.git'
COMMIT='22413752246de3dee8ab0d509f7f7a8889080f95'
MODELS={'LightGCN':'lightgcn','RLMRec-Con':'lightgcn_plus'}
LAYERS=(1,2,3)
SEEDS=(2023,2024,2025)
SEMANTIC_WEIGHT=0.01
# MỖI THÀNH VIÊN TỰ CHỌN 2 GIÁ TRỊ NÀY
MODEL = "LightGCN"      # "LightGCN" hoặc "RLMRec-Con"
LAYER = 1               # 1, 2 hoặc 3
assert MODEL in MODELS, f"MODEL không hợp lệ: {MODEL}"
assert LAYER in LAYERS, f"LAYER không hợp lệ: {LAYER}"
ASSIGNED_MODEL = MODELS[MODEL]
ASSIGNED_LAYER = LAYER
DRY_RUN=True            # Đổi False sau khi PREFLIGHT PASS và kiểm tra manifest
SMOKE_TEST_ONLY=True    # True: run just one pending configuration
ALLOW_CLONE=True
RETRY_FAILED=False  # Giữ False; không ghi đè recovery run đã tồn tại
assert ROOT.is_dir(), ROOT
print('E4 target:',E4,'mode:', 'DRY RUN' if DRY_RUN else 'TRAINING')
print('CẤU HÌNH:', MODEL, ASSIGNED_MODEL, 'layers', ASSIGNED_LAYER, 'seeds', SEEDS)

RECOVERY_ROOT=E4/'recovery_runs'
print('RECOVERY output (new):', RECOVERY_ROOT)
print('Original E4 runs remain untouched:', E4/'runs')


## 1 — Preflight: pin code, validate dataset and configuration

**Important:** `layer_num` is changed *only in an isolated copy* of the repository. E1/E3C files are not modified. We verify that both model YAMLs expose the Amazon graph-layer setting; if not, stop and inspect source before proceeding.

In [ ]:
def git(*args,cwd=None):
    p=subprocess.run(['git',*args],cwd=cwd,text=True,capture_output=True)
    if p.returncode: raise RuntimeError(p.stderr[-2500:])
    return p.stdout.strip()
if not SOURCE.exists():
    if not ALLOW_CLONE: raise FileNotFoundError(SOURCE)
    git('clone',REPO_URL,str(SOURCE))
    git('checkout','--detach',COMMIT,cwd=SOURCE)
assert git('rev-parse','HEAD',cwd=SOURCE)==COMMIT, 'Repository commit differs: do not silently switch.'
assert not git('status','--porcelain',cwd=SOURCE), 'Source is dirty; use a clean runtime.'
required=['trn_mat.pkl','val_mat.pkl','tst_mat.pkl','usr_emb_np.pkl','itm_emb_np.pkl']
assert all((DATA/f).is_file() for f in required), [f for f in required if not (DATA/f).is_file()]
link=SOURCE/'data/amazon'; link.parent.mkdir(parents=True,exist_ok=True)
if not link.exists() and not link.is_symlink(): link.symlink_to(DATA,target_is_directory=True)
assert link.resolve()==DATA.resolve(), 'Dataset path is not the approved Drive split.'
BASE_CONFIG={}
for label,model in MODELS.items():
    path=SOURCE/f'encoder/config/modelconf/{model}.yml'
    assert path.is_file(), f'Missing config: {path}'
    cfg=yaml.safe_load(path.read_text())
    assert 'model' in cfg and 'train' in cfg
    assert 'amazon' in cfg['model'] and 'layer_num' in cfg['model']['amazon'], f'Inspect layer key for {model}'
    assert int(cfg['model']['amazon']['layer_num'])==3, f'Unexpected E1 baseline layer count: {model}'
    if model=='lightgcn_plus':
        assert abs(float(cfg['model']['amazon']['kd_weight'])-SEMANTIC_WEIGHT)<1e-12
    BASE_CONFIG[model]=cfg
print('PREFLIGHT PASS: pinned code, dataset, 3-layer reference, model config.')
try:
    import torch
    print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
except ImportError: print('PyTorch not installed')


## 2 — Manifest: tình trạng bản gốc và thư mục recovery

`runs/` là lịch sử train cũ, **chỉ đọc**. `recovery_runs/` dành riêng cho các lần train lại cần checkpoint. Không xóa thư mục nào.


In [ ]:
def run_dir(model,layer,seed):
    return E4/'runs'/model/f'layers_{layer}'/f'seed_{seed}'
def recovery_dir(model,layer,seed):
    return RECOVERY_ROOT/model/f'layers_{layer}'/f'seed_{seed}'
def state(model,layer,seed):
    d=run_dir(model,layer,seed)
    p=d/'train_status.json'
    if p.is_file():
        return json.loads(p.read_text()).get('status','unknown')
    return 'pending' if not d.exists() else 'unverified_existing_directory'
def recovery_state(model,layer,seed):
    d=recovery_dir(model,layer,seed)
    if not d.exists(): return 'not_started'
    p=d/'train_status.json'
    return json.loads(p.read_text()).get('status','unknown') if p.is_file() else 'unverified_existing_directory'
rows=[]
for label,model in MODELS.items():
    for layer in LAYERS:
        for seed in SEEDS:
            rows.append({'label':label,'model':model,'layers':layer,'seed':seed,
                         'old_status':state(model,layer,seed),
                         'recovery_status':recovery_state(model,layer,seed),
                         'old_checkpoint':len(list((run_dir(model,layer,seed)/'checkpoints').glob('*.pth')))})
manifest=pd.DataFrame(rows)
display(manifest)
print('Original runs:',len(manifest),'| recovery pending:',sum(manifest.recovery_status=='not_started'))


## 3 — Train có checkpoint ghi **trực tiếp lên Drive**

E1 dùng symlink `REPO/encoder/checkpoint/MODEL -> OUTPUT_DIR/checkpoints`. Ở E4, mỗi workspace có symlink tương ứng trỏ tới `recovery_runs/<model>/layers_<L>/seed_<seed>/checkpoints`. Chỉ công nhận run khi `.pth` tồn tại, dung lượng hợp lệ, `torch.load` đọc được, SHA256 xác minh được và `config_snapshot.yml` đã lưu.

**Quan trọng:** trainer phải dùng đúng `cwd=work` và checkpoint mặc định `encoder/checkpoint/<model>/<model>-amazon-<seed>.pth`. Nếu source thay đổi, preflight/smoke test phải phát hiện trước khi chạy cả batch.


In [ ]:
import traceback
from pathlib import Path

def write_json(p,obj):
    p.parent.mkdir(parents=True,exist_ok=True)
    tmp=p.with_name(p.name+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
    tmp.replace(p)

def sha256_file(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):
            h.update(block)
    return h.hexdigest()

def run_one(model,layer,seed):
    import torch
    assert torch.cuda.is_available(), 'Enable GPU runtime.'
    assert model in MODELS.values() and layer in LAYERS and seed in SEEDS
    assert ROOT.is_dir() and DATA.is_dir(), 'Google Drive/dataset not mounted.'
    dest=recovery_dir(model,layer,seed)
    assert not dest.exists(), f'Recovery run already exists; DO NOT overwrite: {dest}'
    work=Path('/content')/f'e4_recovery_work_{model}_L{layer}_s{seed}'
    assert not work.exists(), f'Workspace exists: {work}'

    # Same isolated-source strategy as original E4; exclude stale outputs.
    def ignore(src,names):
        return set(names)&{'.git','data','checkpoint','checkpoints','log','logs','__pycache__','.ipynb_checkpoints'}
    shutil.copytree(SOURCE,work,ignore=ignore,symlinks=True)
    (work/'data').symlink_to(SOURCE/'data',target_is_directory=True)
    config_file=work/f'encoder/config/modelconf/{model}.yml'
    cfg=yaml.safe_load(config_file.read_text())
    cfg['model']['amazon']['layer_num']=int(layer)
    if 'layer_num' in cfg['model']: cfg['model']['layer_num']=int(layer)
    if model=='lightgcn_plus':
        cfg['model']['amazon']['kd_weight']=SEMANTIC_WEIGHT
        cfg['model']['kd_weight']=SEMANTIC_WEIGHT
    cfg['train']['seed']=int(seed)
    cfg['train']['save_model']=True
    config_file.write_text(yaml.safe_dump(cfg,sort_keys=False),encoding='utf-8')

    # E1 technique: checkpoint directory is a SYMLINK to Google Drive.
    # Unique Drive target per (model, layer, seed); never overwrite E4/runs.
    dest.mkdir(parents=True,exist_ok=False)
    cp_drive=dest/'checkpoints'
    cp_drive.mkdir(parents=True,exist_ok=False)
    cp_src=work/'encoder'/'checkpoint'/model
    cp_src.parent.mkdir(parents=True,exist_ok=True)
    assert not cp_src.exists() and not cp_src.is_symlink(), f'Unexpected checkpoint directory: {cp_src}'
    cp_src.symlink_to(cp_drive,target_is_directory=True)
    assert cp_src.resolve()==cp_drive.resolve(), 'Checkpoint symlink not pointing to Drive!'

    (dest/'config_snapshot.yml').write_text(config_file.read_text(),encoding='utf-8')
    meta={'experiment':'E4_Graph_Depth_Recovery','model':model,'layers':layer,'seed':seed,
          'source_commit':COMMIT,'semantic_weight':SEMANTIC_WEIGHT if model=='lightgcn_plus' else None,
          'original_run':str(run_dir(model,layer,seed)),
          'checkpoint_storage':'direct_drive_symlink',
          'status':'running','started_utc':datetime.now(timezone.utc).isoformat()}
    write_json(dest/'train_status.json',meta)
    expected=cp_drive/f'{model}-amazon-{seed}.pth'
    assert not expected.exists(), f'Unexpected existing checkpoint: {expected}'
    cmd=[sys.executable,'encoder/train_encoder.py','--model',model,'--dataset','amazon',
         '--device','cuda','--cuda','0','--seed',str(seed)]
    try:
        with (dest/'training_console.log').open('w',encoding='utf-8') as f:
            proc=subprocess.Popen(cmd,cwd=work,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                                  text=True,errors='replace',bufsize=1)
            for line in proc.stdout:
                f.write(line); f.flush()
                if any(x in line for x in ('Epoch','Recall','NDCG','Error','Traceback','Best')):
                    print(line.rstrip(),flush=True)
            rc=proc.wait()
        meta['exit_code']=rc
        if rc!=0: raise RuntimeError(f'Trainer exit {rc}; see {dest}/training_console.log')
        if not expected.is_file():
            raise RuntimeError(f'NO CHECKPOINT ON DRIVE: {expected}. Workspace retained for debugging.')
        size=expected.stat().st_size
        if size<1024:
            raise RuntimeError(f'Checkpoint suspiciously small: {size} bytes')
        # Same validation principle as E1 Cell 5; only load trusted group-owned checkpoints.
        state_dict=torch.load(expected,map_location='cpu',weights_only=True)
        if not isinstance(state_dict,dict) or not state_dict:
            raise RuntimeError('Checkpoint is empty or invalid state_dict')
        if not all(isinstance(v,torch.Tensor) for v in state_dict.values()):
            raise RuntimeError('Checkpoint contains non-tensor state_dict entries')
        del state_dict
        meta['checkpoint_count']=1
        meta['checkpoint_path']=str(expected)
        meta['checkpoint_bytes']=size
        meta['checkpoint_sha256']=sha256_file(expected)
        meta['status']='trained_pending_export'
        write_json(dest/'checkpoint_verification.json',{
            'model':model,'layer':layer,'seed':seed,'checkpoint_path':str(expected),
            'bytes':size,'sha256':meta['checkpoint_sha256'],'loadable':True,
            'verified_at_utc':datetime.now(timezone.utc).isoformat()})
        print('CHECKPOINT VERIFIED ON DRIVE:',expected,'bytes=',size)
    except Exception as exc:
        meta['status']='failed_checkpoint_or_training'
        meta['error']=repr(exc)
        print('FAILED; original run untouched; workspace retained:',work)
        raise
    finally:
        meta['finished_utc']=datetime.now(timezone.utc).isoformat()
        write_json(dest/'train_status.json',meta)
        # Never delete workspace automatically. Drive persistence is mandatory.
    return meta


### Chạy an toàn

1. Mount Drive, chọn `MODEL`, `LAYER` ở Cell 2; chạy Cell 1–8.
2. Kiểm tra manifest và GPU. Đổi `DRY_RUN=False`, giữ `SMOKE_TEST_ONLY=True` để train **1 seed**.
3. Sau run đầu, kiểm tra `CHECKPOINT VERIFIED ON DRIVE` và `checkpoint_verification.json`.
4. Nếu PASS, đổi `SMOKE_TEST_ONLY=False` và chạy lại Cell 10 để train **các seed recovery chưa có** cho cùng model/layer.
5. **Không xóa** `experiments/E4/runs` và không chuyển trạng thái run cũ. `recovery_runs` là bằng chứng riêng; exporter cần đọc checkpoint ở đây.


In [ ]:
# EXECUTION GATE — DRY_RUN=True by default. Never overwrite existing recovery runs.
manifest=pd.DataFrame([{'model':m,'layers':l,'seed':s,
                        'old_status':state(m,l,s),'recovery_status':recovery_state(m,l,s)}
                       for m in MODELS.values() for l in LAYERS for s in SEEDS])
if DRY_RUN:
    print('SAFE DRY RUN: no training. Change DRY_RUN=False in Cell 2 after preflight.')
else:
    pending=[(r.model,int(r.layers),int(r.seed)) for r in manifest.itertuples()
             if r.recovery_status=='not_started' and r.model==ASSIGNED_MODEL
             and int(r.layers)==ASSIGNED_LAYER]
    if SMOKE_TEST_ONLY: pending=pending[:1]
    print('TRAIN RECOVERY ONLY:',MODEL,'layer',LAYER,'queue:',pending)
    if not pending: print('No new recovery runs pending; existing files untouched.')
    for m,l,s in pending:
        print(run_one(m,l,s))


## 4 — Kiểm tra checkpoint trên Drive (read-only)

Một run chỉ được coi là đủ điều kiện export khi `recovery_runs/.../checkpoints/*.pth` tồn tại, `checkpoint_verification.json` ghi nhận hash và `train_status.json` có trạng thái `trained_pending_export`. Notebook này **không export Top-K**.


In [ ]:
records=[]
for m in MODELS.values():
    for l in LAYERS:
        for s in SEEDS:
            d=recovery_dir(m,l,s)
            cp=d/'checkpoints'/f'{m}-amazon-{s}.pth'
            vp=d/'checkpoint_verification.json'
            st=recovery_state(m,l,s)
            verified=vp.is_file() and cp.is_file() and cp.stat().st_size>1024
            records.append({'model':m,'layer':l,'seed':s,'old_status':state(m,l,s),
                            'recovery_status':st,'checkpoint_on_drive':cp.is_file(),
                            'verified':verified,'ready_for_export':verified and st=='trained_pending_export'})
readiness=pd.DataFrame(records)
display(readiness)
print('READY:',int(readiness.ready_for_export.sum()),'/18')
